In [ ]:
import torch
import matplotlib.pyplot as plt
from PolicyIteration_put import Policy_Iteration as PI


## Run PI

In [ ]:
# Parameters
d = 1  # dimension
total_time = 1.0
n_time_steps = 50  # number of time grid points
dt = total_time / n_time_steps # width of time mesh
r = 0.06  # interest rate
sigma = 0.4  # volatility
strike = 40.0  # strike price
x_0 = 40.0  # initial price
train_batch_size = 2**10

# Use double precision
torch.set_default_dtype(torch.float64)
device = torch.device('cpu')


### PI with Convergence evaluation, $\lambda=1,K=10 $ (can adjust $\lambda $ and $K$)

In [ ]:
lambda_temp = 1  # temperature parameter
K = 10.0  # penalty factor

PE_iteration = 1000
PI_iteration = 10

epsilon_list = [0.0, 0.2, 0.4]

solver_list_1 = [PI(
    d=d,
    total_time=total_time,
    n_time_steps=n_time_steps,
    K=K,
    r=r,
    sigma=sigma,
    strike=strike,
    x_0=x_0,
    lambda_temp=lambda_temp,
    epsilon = epsilon,
    device = device,
    hidden_layers=2,
    hidden_dim=21,
    lr=0.001,
)
    for epsilon in epsilon_list ]


for i, epsilon in enumerate(epsilon_list):
    solver_list_1[i].PolicyIteration(
        PI_iteration=PI_iteration,
        PE_iteration=PE_iteration,
        batch_size=train_batch_size,
    )


### Plot $Y_0 $ Convergence

In [ ]:
import seaborn as sns
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

epsilon_colors = [
'#1f77b4',  # Blue
'#ff7f0e',   # Orange
'#2ca02c',   # Green
]

eps_ref_value = [
    5.302,
    4.420,
    3.725,
]

def plot_y0_evolution( iteration=PI_iteration - 1):
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    iterations = list(range(1, iteration + 2))
    for i, PI_solver in enumerate(solver_list_1):
        y0_vals = PI_solver.pi_history['y0_values'][:iteration + 1]
        label = f'v(0,40) (ε={epsilon_list[i]})'
        ax.plot(iterations, y0_vals,
                marker='o', markersize=8, linewidth=5, color=epsilon_colors[i] , alpha=0.6,
                label=label)
        ax.axhline(y=eps_ref_value[i], color=epsilon_colors[i], linestyle='--',
                label=f'Reference value (ε={epsilon_list[i]})', alpha=0.6, linewidth=3)

    ax.set_xlabel('PI Step', fontsize=37)
    ax.set_ylabel('v', fontsize=37)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='lower right', fontsize=27)
    ax.tick_params(axis='x', labelsize=35)
    ax.tick_params(axis='y', labelsize=35)

    plt.tight_layout()

    # Save figure
    filename = f'put_Y0_evolution_comparison_lambda_{lambda_temp}_penalty_{K}.pdf'
    plt.savefig(filename, format='pdf', bbox_inches='tight')
    print(f"\nSaved comparison plot: {filename}")

    plt.show()
    plt.close()

plot_y0_evolution()
